# LLMServeLab on Colab (T4)

Runs the GPU exit checks: vLLM 0.10.2 serving `Qwen/Qwen2.5-7B-Instruct-AWQ` on one Tesla T4, then `env_check`, `smoke` and `llmserve benchmark`.

Before running: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Notes and traps: [`docs/runbooks/colab.md`](https://github.com/vedantpople4/llmservelab/blob/main/docs/runbooks/colab.md).

Every step is timed. Its output goes to a log file, and logs, `timings.jsonl`, `vllm.log` and `results/` are copied to `MyDrive/llmservelab/sessions/<UTC timestamp>/` after each step, so a disconnect loses at most the step in progress.

## 0. Parameters

`QUICK = True` checks the whole pipeline in about 15 minutes on a fresh session (20 smoke requests, `configs/dev/e01_quick.yaml`). `QUICK = False` runs the real exit checks (100 smoke requests, `configs/baseline/e01.yaml`, about 1 to 1.5 hours). Only `QUICK = False` runs count as recorded results.

`REPO_REF` is the branch to clone. Use `main` once the Colab PR is merged.

In [ ]:
QUICK = True
REPO_REF = "main"

SMOKE_REQUESTS = 20 if QUICK else 100
BENCHMARK_CONFIG = "configs/dev/e01_quick.yaml" if QUICK else "configs/baseline/e01.yaml"
MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"
MODEL_REVISION = "b25037543e9394b818fdfca67ab2a00ecc7dd641"  # = server.model_revision in configs

## 1. Drive, logging helpers, quiet settings

In [ ]:
import contextlib
import json
import os
import shutil
import subprocess
import time
from datetime import UTC, datetime

from google.colab import drive

drive.mount("/content/drive")
SESSION = f"/content/drive/MyDrive/llmservelab/sessions/{datetime.now(UTC):%Y%m%dT%H%M%SZ}"
REPO = "/content/llmservelab"
LOGS = "/content/logs"
VLLM_LOG = "/content/vllm.log"
os.makedirs(SESSION, exist_ok=True)
os.makedirs(LOGS, exist_ok=True)

os.environ.update(
    {
        "PYTHONWARNINGS": "ignore::DeprecationWarning,ignore::FutureWarning",
        "HF_HUB_DISABLE_PROGRESS_BARS": "1",
        "TOKENIZERS_PARALLELISM": "false",
        "UV_NO_PROGRESS": "1",
    }
)


def save():
    """Copy local logs and results to the Drive session folder; called between steps only."""
    shutil.copytree(LOGS, f"{SESSION}/logs", dirs_exist_ok=True)
    if os.path.exists(VLLM_LOG):
        shutil.copy(VLLM_LOG, f"{SESSION}/vllm.log")
    if os.path.isdir(f"{REPO}/results"):
        shutil.copytree(f"{REPO}/results", f"{SESSION}/results", dirs_exist_ok=True)


@contextlib.contextmanager
def timed(step, **extra):
    """Time one step, append it to timings.jsonl, and save to Drive even if the step fails."""
    started = datetime.now(UTC)
    t0 = time.monotonic()
    print(f"[{started:%H:%M:%S}Z] {step}: start")
    status = "ok"
    try:
        yield
    except BaseException:
        status = "failed"
        raise
    finally:
        elapsed = time.monotonic() - t0
        row = {"step": step, "started_utc": started.isoformat(), "elapsed_s": round(elapsed, 1)}
        row |= {"status": status, "quick": QUICK, **extra}
        with open(f"{LOGS}/timings.jsonl", "a") as f:
            f.write(json.dumps(row) + "\n")
        print(f"[{datetime.now(UTC):%H:%M:%S}Z] {step}: {status} in {elapsed / 60:.1f} min")
        save()


def run(step, cmd, check=True):
    """Run a shell command as a timed step; output goes to the cell and logs/<step>.log."""
    with timed(step, cmd=cmd):
        cwd = REPO if os.path.isdir(REPO) else "/content"
        with open(f"{LOGS}/{step}.log", "w") as log:
            proc = subprocess.Popen(
                cmd,
                shell=True,
                cwd=cwd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )
            for line in proc.stdout:
                print(line, end="")
                log.write(line)
            code = proc.wait()
        if check and code != 0:
            raise RuntimeError(f"{step} exited {code}; see {SESSION}/logs/{step}.log")
        return code


print("session folder:", SESSION)

## 2. Environment (about 3 to 6 min)

In [ ]:
run(
    "1_setup",
    f"git clone --branch {REPO_REF} https://github.com/vedantpople4/llmservelab.git {REPO}"
    f" && cd {REPO} && git log --oneline -1"
    " && nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv"
    " && pip install -q --root-user-action=ignore uv"
    " && uv sync --locked --extra dev --extra mock --extra gpu",
)
os.chdir(REPO)

`nvidia-smi` must say `Tesla T4`. Any other GPU is a different SKU: change the runtime type and reconnect.

## 3. vLLM server (§6 flags, identical to Kaggle and `docker/compose.yml`; about 4 to 10 min first time)

In [ ]:
import urllib.request

with timed("2_server_start"):
    server = subprocess.Popen(
        [
            "uv",
            "run",
            "vllm",
            "serve",
            MODEL,
            "--revision",
            MODEL_REVISION,
            "--host",
            "127.0.0.1",
            "--port",
            "8000",
            "--max-model-len",
            "10240",
            "--gpu-memory-utilization",
            "0.90",
            "--dtype",
            "float16",
            "--no-enable-prefix-caching",
            "--max-num-seqs",
            "256",
            "--disable-log-requests",
        ],
        stdout=open(VLLM_LOG, "w"),
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )
    for _ in range(180):  # first start downloads ~6 GB of weights
        try:
            urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)
            print("vLLM healthy")
            break
        except Exception:
            if server.poll() is not None:
                break
            time.sleep(5)
    else:
        server.terminate()
    if server.poll() is not None:
        print(open(VLLM_LOG).read()[-4000:])
        raise RuntimeError("vLLM did not become healthy")

run("2b_vllm_log_check", f"grep -E 'V1 engine|V0|bfloat16|float16' {VLLM_LOG} | head", check=False)

## 4. Env check and smoke (env check about 30 s; smoke about 1.5 min quick, 5 to 8 min full)

In [ ]:
run("3_env_check", "uv run python -m llmserve.runner.env_check configs/baseline/e01.yaml")
run(
    "3b_smoke",
    "uv run python scripts/smoke.py configs/baseline/e01.yaml"
    f" --requests {SMOKE_REQUESTS} --skip-env-check",
)

Smoke's summary prints wall time for its requests. A full benchmark (5 reps × 200 requests) takes about 10× the time of a 100-request smoke.

## 5. Benchmark (about 3 min quick, 50 to 80 min full)

In [ ]:
run("4_benchmark", f"uv run llmserve benchmark {BENCHMARK_CONFIG}")

## 5b. Phase 3 suite (optional; about 20 min to 3.5 h per suite)

Set `SUITE` to a file from `configs/suites/` (see `docs/runbooks/phase3.md` for the order). Suite results go straight to `MyDrive/llmservelab/results/`, a fixed folder shared by every session, so a suite interrupted by a disconnect resumes in the next session: finished points are skipped. Results are written only after each repetition, never during requests. `SUITE_ARGS` takes `--only LABEL ...`, `--set load.capacity_rps=<req/s>` (H2 suites) or `--plan`.

In [ ]:
SUITE = None  # e.g. "configs/suites/prompt_length.yaml"
SUITE_ARGS = ""  # e.g. "--only p0128 p0256" or "--set load.capacity_rps=1.4"
DRIVE_RESULTS = "/content/drive/MyDrive/llmservelab/results"

if SUITE:
    os.makedirs(DRIVE_RESULTS, exist_ok=True)
    name = os.path.splitext(os.path.basename(SUITE))[0]
    run(f"5_suite_{name}", f"uv run llmserve suite {SUITE} -o {DRIVE_RESULTS} {SUITE_ARGS}")
    run("5b_report", f"uv run llmserve report {DRIVE_RESULTS} -o {DRIVE_RESULTS}/report")

## 6. Timing summary

In [ ]:
with open(f"{LOGS}/timings.jsonl") as f:
    rows = [json.loads(line) for line in f]
for r in rows:
    print(f"{r['step']:<20} {r['status']:<7} {r['elapsed_s'] / 60:6.1f} min")
print(f"{'total':<28} {sum(r['elapsed_s'] for r in rows) / 60:6.1f} min")
save()
print("saved to", SESSION)

When done measuring: **Runtime → Disconnect and delete runtime**, so the free GPU quota stops draining.